# 🛡️ Pipeline de Prevención de Fraude - Documentación Técnica
**Candidato:** [Tu Nombre]  
**Fecha:** 23 de junio de 2026

## 1. Introducción
Este notebook detalla el diseño y la implementación de un pipeline ETL (Extracción, Transformación y Carga) para el procesamiento de transacciones bancarias, integrando una lógica de detección de anomalías de gasto.

La solución cumple con tres objetivos principales:
1. **Calidad de Datos:** Limpieza y normalización de registros crudos.
2. **Persistencia:** Carga centralizada en PostgreSQL (Supabase).
3. **Analítica:** Detección de patrones de fraude mediante SQL Avanzado (Window Functions).

## 2. Fase 1: ETL (Pandas)
La arquitectura de procesamiento se basa en **Pandas** por su capacidad de vectorización, garantizando eficiencia.

* **Deduplicación:** Se priorizó el `id_transaccion` como llave única, manteniendo solo la primera ocurrencia.
* **Imputación de Nulos:** La regla de negocio de imputar `0.0` en transacciones rechazadas con monto nulo asegura la trazabilidad del intento de fraude sin sesgar promedios financieros.
* **Ingeniería de Características:** Se creó la columna booleana `es_monto_inusual`, optimizando la consulta posterior.

In [1]:
import pandas as pd
# Simulamos la carga para verificar la estructura
df = pd.read_csv('../data/transacciones_diarias.csv')
df_clean = df.drop_duplicates(subset=['id_transaccion'], keep='first')
print(f"Registros procesados: {len(df_clean)}")
print(df_clean.head())

Registros procesados: 122
  id_transaccion id_cliente           fecha_hora  monto_usd  tipo_comercio  \
0          T-001      C-101  2026-06-18 08:00:00       50.0       nacional   
2          T-002      C-102  2026-06-18 08:15:00        NaN  internacional   
3          T-003      C-103  2026-06-18 08:20:00      120.5       nacional   
4          T-004      C-104  2026-06-18 08:45:00     1850.0  internacional   
5          T-005      C-101  2026-06-18 09:30:00      350.0  internacional   

  estado_transaccion  
0           aprobada  
2          rechazada  
3          pendiente  
4           aprobada  
5           aprobada  


## 3. Fase 2: Análisis SQL de Anomalías
Para la detección de fraude (transacciones > 5x la anterior), delegamos la lógica al motor de base de datos. 

* **CTE (Common Table Expression):** Aísla las transacciones aprobadas antes de aplicar la función `LAG()`.
* **Window Functions:** La función `LAG()` permite comparar cronológicamente la transacción actual con la anterior por cliente (`PARTITION BY id_cliente`), minimizando el uso de memoria en la aplicación.

![Resultados SQL](assets/captura_sql_resultados.png)

In [6]:
%%sql
WITH TransaccionesAprobadas AS (
    SELECT 
        id_cliente, monto_usd, fecha_hora,
        LAG(monto_usd) OVER (PARTITION BY id_cliente ORDER BY fecha_hora) as monto_anterior
    FROM transacciones
    WHERE estado_transaccion = 'aprobada'
)
SELECT * FROM TransaccionesAprobadas 
WHERE monto_usd >= (monto_anterior * 5);

UsageError: Cell magic `%%sql` not found.


## 4. Fase 3: Orquestación (Airflow)
El flujo fue diseñado para ejecutarse diariamente a las 11:30 PM. 

* **Dependencia:** El operador `tarea_analitica` solo se activa si `tarea_etl` finaliza con estado exitoso.
* **Resiliencia:** Se configuraron políticas de reintento (`retries: 1`) ante posibles fallos intermitentes de red con la base de datos Supabase.

## 5. Conclusiones y Futuros Pasos
Este pipeline MVP proporciona una base sólida para la prevención de fraude. 

* **Próximas mejoras:**
  - Implementar **Validación de Esquema** con `Great Expectations`.
  - Configurar **alertas en tiempo real** (Slack/Email) en Airflow ante anomalías detectadas.
  - Migrar los secretos de conexión a un **Secret Manager** (AWS/GCP).